# Unit 3, Lecture 3: Building on Microsoft Agent Framework

The payoff. Two lectures of foundation, now a **real, shippable agent** on the
successor framework.

The pleasant surprise: a tool here is **just a plain Python function** with type
hints and a docstring. No decorator, no schema dict, no class. You pass it to the
agent and the framework reads the schema off the signature.

Runs against the real `agent-framework` package. Tools run offline; running the
agent needs a lane.

## The tools are plain functions

In [ ]:
from cse476.agent_fw import classify_ticket, get_sla, list_queues, TRIAGE_TOOLS

# no framework, no model, no agent. just call them.
print(classify_ticket("I was charged twice and want a refund"))
print(get_sla("billing"))
print(list_queues())
print()
print("the tool suite is just a list of functions:", [t.__name__ for t in TRIAGE_TOOLS])

Where does the schema come from? The **function name** becomes the tool name,
the **docstring** becomes the description, the **`Annotated` type** becomes the
parameter. You write a normal, well-documented function and the framework has
everything. Good documentation stopped being politeness and became the
interface.

## Test the tools with no framework

Because tools are plain functions, your whole tool layer is testable with
ordinary unit tests, no framework, no model. All your Unit 2 testing skills apply
here unchanged.

In [ ]:
# ordinary asserts, offline, free
assert "billing" in classify_ticket("refund please")
assert "abuse" in classify_ticket("someone hacked my account")
assert "24 hour" in get_sla("billing")
assert "No SLA" in get_sla("nonsense")
print("tool layer green, no model spent")

## The whole agent, in three lines

Now assemble the agent. Read the three arguments and recognise them: the client
is your connector (lane), the instructions are your system prompt, the tools are
your `REGISTRY`. **This cell needs `GITHUB_TOKEN` in `.env`.**

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from cse476.agent_fw import make_client, build_support_agent

client = make_client()               # your lane, framework-shaped
agent  = build_support_agent(client) # instructions + tools, assembled
print("agent built:", type(agent).__name__)

In [ ]:
# await agent.run(...) is your entire Unit 1 for-step-in-range loop
result = await agent.run("I was double charged this month, can you help?")
print(result)

That one awaited call did what your Unit 1 loop did: showed the model your
tool schemas, let it choose `classify_ticket` and `get_sla`, called them, fed the
results back, and produced an answer. **You did not lose the loop. You stopped
having to write it**, and because you wrote it once by hand, you know every step
it just took.

## The mapping, complete

In [ ]:
from cse476.agent_fw import AGENT_FRAMEWORK_MAP

for framework_word, hand_built in AGENT_FRAMEWORK_MAP.items():
    print(f"{framework_word:28} ->  {hand_built}")

## Why `agent.run` is awaited

Unit 1 was synchronous: each model call blocked until it returned, and while it
waited on the network your whole program sat idle. The framework is **async**:
`await` says "pause here until this returns, but let other work run meanwhile".

In [ ]:
from cse476.agent_fw import async_matters

for k, v in async_matters().items():
    print(f"{k:16}: {v}")

This is not just syntax. A synchronous agent handles one request at a time.
An async one can have many agents in flight, each yielding while it waits on its
own model call. That is the difference between a demo and a service.

## The seed of Unit 4: an agent as a tool

A tool can be a function, or it can be **another whole agent**. `agent.as_tool()`
turns an agent into something another agent can call, which is how multi-agent
systems are built. You are seeing the door here; Unit 4 walks through it.

In [ ]:
# a manager agent whose tools are other agents (construction only)
triage = build_support_agent(client)
manager = client.as_agent(
    instructions="You delegate support questions to your specialist agents.",
    tools=[triage.as_tool()],   # the triage agent, used as a tool
)
print("manager built with an agent-as-tool:", type(manager).__name__)

## Your turn

**1. Ship a real agent.** Build an Agent Framework agent with three tools of your
own, all plain functions. Run it on a real question and watch it choose and call
your tools. This is a shippable agent, not a toy.

**2. Test the tools alone.** Write ordinary asserts for your three tool functions,
no framework, no model. Confirm the whole tool layer is green before the agent
ever runs.

**3. Design a manager.** In words, sketch a manager agent that delegates to two
specialist agents via `as_tool`. You will build this in Unit 4; naming the pieces
now makes that build easier.

In [ ]:
# your work here
